# DiD standard and lagged

In [6]:
import pandas as pd
import statsmodels.formula.api as smf
import os
from pandas.tseries.offsets import DateOffset

# ===== Configuration =====
aligned_dirs = {
    "cumulative": {
        "dir": "aligned_did_outputs_v2_cumulative",
        "metrics": ['cumulative_stars', 'cumulative_forks', 'cumulative_commits', 'cumulative_prs', 'cumulative_ve_contributors'],
        "prefix": ""
    },
    "non_cumulative": {
        "dir": "aligned_did_outputs_new",
        "metrics": ['new_stars', 'new_forks', 'new_commits', 'new_prs', 'active_contributors'],
        "prefix": ""
    }
}

# ===== Main Analysis =====
for data_type, config in aligned_dirs.items():
    print(f"\n{'='*50}\nAnalyzing {data_type} metrics\n{'='*50}")

    for metric in config['metrics']:
        # Load data
        file_path = os.path.join(config['dir'], f"aligned_did_{metric}.csv")
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                print(f"[!] Skipping empty file: {file_path}")
                continue
        except FileNotFoundError:
            print(f"[!] File not found: {file_path}")
            continue

        # Prep variables
        outcome = f"{config['prefix']}{metric}" if config['prefix'] else metric
        df['treatment'] = (df['repo_type'] == 'treatment').astype(int)
        df['post'] = df['post_treatment'].astype(int)
        df['did'] = df['treatment'] * df['post']

        # ===== 1. Original DiD =====
        model_did = smf.ols(f"{outcome} ~ treatment + post + did", data=df).fit(cov_type='HC3')
        print(f"\n[Original DiD] {outcome}:")
        print(f"DID coefficient: {model_did.params['did']:.4f} (p={model_did.pvalues['did']:.4f})")

        # ===== 2. Lagged Metric Analysis =====
        df = df.sort_values(by=['repo_full_name', 'month'])
        df['lagged_outcome'] = df.groupby('repo_full_name')[outcome].shift(1)

        if df['lagged_outcome'].notnull().any():  # Only run if lag exists
            model_lag = smf.ols(
                f"{outcome} ~ lagged_outcome + treatment + post + did",
                data=df.dropna(subset=['lagged_outcome'])
            ).fit(cov_type='HC3')
            print(f"\n[Lagged Model] {outcome}:")
            print(f"DID coefficient with lag: {model_lag.params['did']:.4f} (p={model_lag.pvalues['did']:.4f})")

        # ===== 3. Placebo Test (pre-period only) =====
        pre = df[df['relative_month'] < 0].dropna(subset=[outcome]).copy()
        pre['post_placebo'] = (pre['relative_month'] >= -3).astype(int)
        pre['placebo_did'] = pre['treatment'] * pre['post_placebo']

        model_placebo = smf.ols(
            f"{outcome} ~ treatment + post_placebo + placebo_did",
            data=pre
        ).fit(cov_type='cluster', cov_kwds={'groups': pre['pair_id']})

        print(f"\n[Placebo Test] {outcome}:")
        print(f"Placebo DID coefficient: {model_placebo.params['placebo_did']:.4f} (p={model_placebo.pvalues['placebo_did']:.4f})")
        print("-"*80)


Analyzing cumulative metrics

[Original DiD] cumulative_stars:
DID coefficient: 147.6710 (p=0.0000)

[Lagged Model] cumulative_stars:
DID coefficient with lag: 18.9027 (p=0.0011)

[Placebo Test] cumulative_stars:
Placebo DID coefficient: 1.6733 (p=0.0078)
--------------------------------------------------------------------------------

[Original DiD] cumulative_forks:
DID coefficient: 33.5575 (p=0.0000)

[Lagged Model] cumulative_forks:
DID coefficient with lag: 2.9334 (p=0.0228)

[Placebo Test] cumulative_forks:
Placebo DID coefficient: 1.2429 (p=0.0006)
--------------------------------------------------------------------------------

[Original DiD] cumulative_commits:
DID coefficient: 17.9982 (p=0.0113)

[Lagged Model] cumulative_commits:
DID coefficient with lag: 1.0600 (p=0.2230)

[Placebo Test] cumulative_commits:
Placebo DID coefficient: 0.9487 (p=0.0452)
--------------------------------------------------------------------------------

[Original DiD] cumulative_prs:
DID coeffici

# placebo test, Event-study regression

## New metrics
Only new metrics used for pretrend event study Wald's test.

### Data preparation

In [12]:
# Directory containing the aligned files
aligned_dir = "aligned_did_outputs_new"
metrics = ['new_stars', 'new_forks', 'new_commits', 'new_prs', 'active_contributors']
metric_column_map = {
    "new_stars": "new_stars",
    "new_forks": "new_forks",
    "new_commits": "new_commits",
    "new_prs": "new_prs",
    "active_contributors": "active_contributors"
}

results = {}
placebo_results = {}

In [13]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np
from linearmodels import PanelOLS  # For more robust estimation

for metric in metrics:
    file_path = os.path.join(aligned_dir, f"aligned_did_{metric}.csv")
    df = pd.read_csv(file_path, parse_dates=['month', 'hn_submission_date'])

    if df.empty:
        print(f"[!] Skipping empty file: {file_path}")
        continue

    # ========== DATA PREP ==========
    # Create proper time variables
    df['time_period'] = df['month'].astype('str')  # For fixed effects
    df['month_diff'] = df['relative_month']  # Already appears to be months from treatment

    # Treatment indicators
    df['treatment'] = (df['repo_type'] == 'treatment').astype(int)
    df['post'] = df['post_treatment'].astype(int)
    df['did'] = df['treatment'] * df['post']
    outcome = metric_column_map[metric]

### Improved DiD

In [36]:
for metric in metrics:
    # ========== IMPROVED DiD ANALYSIS ==========
    print(f"\n===== Enhanced DiD Analysis: {outcome} =====")

    # Option 1: Two-way fixed effects (more robust)
    try:
        df_panel = df.set_index(['repo_full_name', 'month'])
        model_twfe = PanelOLS.from_formula(
            f"{outcome} ~ treatment + post + did + EntityEffects + TimeEffects",
            data=df_panel
        ).fit(cov_type='clustered', cluster_entity=True)

        print("\nTwo-way FE model results:")
        print(model_twfe.summary)
    except Exception as e:
        print(f"PanelOLS failed: {str(e)}")

    # Option 2: OLS with repo clusters
    model_ols = smf.ols(
        formula=f"{outcome} ~ treatment + post + did + C(pair_id)",
        data=df
    ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

    print("\nOLS with repo clustering:")
    print(model_ols.summary())


===== Enhanced DiD Analysis: cumulative_ve_contributors =====
PanelOLS failed: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          treatment

Set drop_absorbed=True to automatically drop absorbed variables.


OLS with repo clustering:
                                OLS Regression Results                                
Dep. Variable:     cumulative_ve_contributors   R-squared:                       0.731
Model:                                    OLS   Adj. R-squared:                  0.720
Method:                         Least Squares   F-statistic:                     1595.
Date:                        Fri, 09 Oct 2026   Prob (F-statistic):          1.04e-166
Time:      

### Dynamic effects

In [15]:
for metric in metrics:
    file_path = os.path.join(aligned_dir, f"aligned_did_{metric}.csv")
    df = pd.read_csv(file_path, parse_dates=['month', 'hn_submission_date'])
    outcome = metric_column_map[metric]

    df['treatment'] = (df['repo_type'] == 'treatment').astype(int)
    df['rel_month_cat'] = df['relative_month'].astype(int)
    ref_period = -1

    event_model = smf.ols(
        f"{outcome} ~ C(rel_month_cat, Treatment(reference={ref_period}))*treatment",
        data=df
    ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

    print(f"\n===== Event study: {outcome} =====")
    print(event_model.summary().tables[1])

    pre_terms = [
        p for p in event_model.params.index
        if ':treatment' in p and int(p.split('[T.')[1].split(']')[0]) < ref_period
    ]
    if pre_terms:
        wald = event_model.wald_test(', '.join(pre_terms), scalar=True)
        print(f"\nJoint pre-trend test (leads jointly = 0?): {wald}")


===== Event study: new_stars =====
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
Intercept                                                     4.2931      1.145      3.750      0.000       2.049       6.537
C(rel_month_cat, Treatment(reference=-1))[T.-6]              -0.5172      0.679     -0.762      0.446      -1.848       0.813
C(rel_month_cat, Treatment(reference=-1))[T.-5]              -0.4828      0.551     -0.876      0.381      -1.563       0.598
C(rel_month_cat, Treatment(reference=-1))[T.-4]              -0.2069      0.390     -0.530      0.596      -0.972       0.558
C(rel_month_cat, Treatment(reference=-1))[T.-3]              -0.4138      0.428     -0.966      0.334      -1.254       0.426
C(rel_month_cat, Treatment(reference=-1))[T.-2]              -0.6897      0.587   

### Placebo test

In [16]:
for metric in metrics:
    file_path = os.path.join(aligned_dir, f"aligned_did_{metric}.csv")
    df = pd.read_csv(file_path, parse_dates=['month', 'hn_submission_date'])

    if df.empty:
        print(f"[!] Skipping empty file: {file_path}")
        continue
    print(event_model.summary().tables[1])

    # ========== PLACEBO TESTS ==========
    print(f"\n----- Placebo Tests for {outcome} -----")

    # Pre-treatment placebos (should show no effect)
    for months_before in [2, 4]:
        df[f'placebo_{months_before}'] = (df['month_diff'] == -months_before).astype(int)
        df[f'did_placebo_{months_before}'] = df['treatment'] * df[f'placebo_{months_before}']

        placebo_model = smf.ols(
            f"{outcome} ~ treatment + placebo_{months_before} + did_placebo_{months_before} + C(pair_id)",
            data=df
        ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

        print(f"\nPlacebo {months_before} months before treatment:")
        print(placebo_model.summary().tables[1])

                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
Intercept                                                     2.7019      1.240      2.180      0.029       0.272       5.132
C(rel_month_cat, Treatment(reference=-1))[T.-6]              -0.1154      0.147     -0.786      0.432      -0.403       0.172
C(rel_month_cat, Treatment(reference=-1))[T.-5]              -0.2115      0.208     -1.018      0.309      -0.619       0.196
C(rel_month_cat, Treatment(reference=-1))[T.-4]              -0.2500      0.117     -2.132      0.033      -0.480      -0.020
C(rel_month_cat, Treatment(reference=-1))[T.-3]              -0.1923      0.131     -1.470      0.141      -0.449       0.064
C(rel_month_cat, Treatment(reference=-1))[T.-2]               0.1154      0.170      0.677      0.499      -0.219     

KeyError: 'month_diff'

## Culumative metrics
Only for exploration, not used for pretrend analysis

### data preparation

In [27]:
### Data preparation
# Directory containing the aligned files
# Set up
aligned_dir = "aligned_did_outputs_v2_cumulative"
metrics = ['cumulative_stars', 'cumulative_forks', 'cumulative_commits', 'cumulative_prs', 'cumulative_ve_contributors']
metric_column_map = {
    "cumulative_stars": "cumulative_stars",
    "cumulative_forks": "cumulative_forks",
    "cumulative_commits": "cumulative_commits",
    "cumulative_prs": "cumulative_prs",
    "cumulative_ve_contributors": "cumulative_ve_contributors"
}

results = {}
placebo_results = {}

In [28]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np
from linearmodels import PanelOLS  # For more robust estimation

for metric in metrics:
    file_path = os.path.join(aligned_dir, f"aligned_did_{metric}.csv")
    df = pd.read_csv(file_path, parse_dates=['month', 'hn_submission_date'])

    if df.empty:
        print(f"[!] Skipping empty file: {file_path}")
        continue

    # ========== DATA PREP ==========
    # Create proper time variables
    df['time_period'] = df['month'].astype('str')  # For fixed effects
    df['month_diff'] = df['relative_month']  # Already appears to be months from treatment

    # Treatment indicators
    df['treatment'] = (df['repo_type'] == 'treatment').astype(int)
    df['post'] = df['post_treatment'].astype(int)
    df['did'] = df['treatment'] * df['post']
    outcome = metric_column_map[metric]

### improved DiD

In [ ]:
### Improved DiD
for metric in metrics:
    # ========== IMPROVED DiD ANALYSIS ==========
    print(f"\n===== Enhanced DiD Analysis: {outcome} =====")

    # Option 1: Two-way fixed effects (more robust)
    try:
        df_panel = df.set_index(['repo_full_name', 'month'])
        model_twfe = PanelOLS.from_formula(
            f"{outcome} ~ treatment + post + did + EntityEffects + TimeEffects",
            data=df_panel
        ).fit(cov_type='clustered', cluster_entity=True)

        print("\nTwo-way FE model results:")
        print(model_twfe.summary)
    except Exception as e:
        print(f"PanelOLS failed: {str(e)}")

    # Option 2: OLS with repo clusters
    model_ols = smf.ols(
        formula=f"{outcome} ~ treatment + post + did + C(pair_id)",
        data=df
    ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

    print("\nOLS with repo clustering:")
    print(model_ols.summary())


===== Enhanced DiD Analysis: cumulative_ve_contributors =====
PanelOLS failed: Unable to evaluate factor `treatment`. [NameError: `treatment` is not present in the dataset or evaluation context.]


PatsyError: Error evaluating factor: NameError: name 'post' is not defined
    cumulative_ve_contributors ~ treatment + post + did + C(pair_id)
                                             ^^^^

### dynamic effects

In [30]:
### Dynamic effects
for metric in metrics:
    file_path = os.path.join(aligned_dir, f"aligned_did_{metric}.csv")
    df = pd.read_csv(file_path, parse_dates=['month', 'hn_submission_date'])
    outcome = metric_column_map[metric]

    df['treatment'] = (df['repo_type'] == 'treatment').astype(int)
    df['rel_month_cat'] = df['relative_month'].astype(int)
    ref_period = -1

    event_model = smf.ols(
        f"{outcome} ~ C(rel_month_cat, Treatment(reference={ref_period}))*treatment",
        data=df
    ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

    print(f"\n===== Event study: {outcome} =====")
    print(event_model.summary().tables[1])

    pre_terms = [
        p for p in event_model.params.index
        if ':treatment' in p and int(p.split('[T.')[1].split(']')[0]) < ref_period
    ]
    if pre_terms:
        wald = event_model.wald_test(', '.join(pre_terms), scalar=True)
        print(f"\nJoint pre-trend test (leads jointly = 0?): {wald}")


===== Event study: cumulative_stars =====
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
Intercept                                                    40.5800      8.491      4.779      0.000      23.938      57.222
C(rel_month_cat, Treatment(reference=-1))[T.-6]             -11.8000      2.366     -4.988      0.000     -16.437      -7.163
C(rel_month_cat, Treatment(reference=-1))[T.-5]              -9.8200      2.024     -4.852      0.000     -13.786      -5.854
C(rel_month_cat, Treatment(reference=-1))[T.-4]              -7.4200      1.665     -4.456      0.000     -10.684      -4.156
C(rel_month_cat, Treatment(reference=-1))[T.-3]              -4.6000      1.189     -3.868      0.000      -6.931      -2.269
C(rel_month_cat, Treatment(reference=-1))[T.-2]              -2.3200      0

### placebo test

In [37]:
### Placebo test
for metric in metrics:
    # ========== PLACEBO TESTS ==========
    print(f"\n----- Placebo Tests for {outcome} -----")

    # Pre-treatment placebos (should show no effect)
    for months_before in [2, 4]:
        df[f'placebo_{months_before}'] = (df['month_diff'] == -months_before).astype(int)
        df[f'did_placebo_{months_before}'] = df['treatment'] * df[f'placebo_{months_before}']

        placebo_model = smf.ols(
            f"{outcome} ~ treatment + placebo_{months_before} + did_placebo_{months_before} + C(pair_id)",
            data=df
        ).fit(cov_type='cluster', cov_kwds={'groups': df['repo_full_name']})

        print(f"\nPlacebo {months_before} months before treatment:")
        print(placebo_model.summary().tables[1])


----- Placebo Tests for cumulative_ve_contributors -----

Placebo 2 months before treatment:
                                                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                                                         4.6268      0.618      7.489      0.000       3.416       5.838
C(pair_id)[T.Avaiga/taipy__vs__GoogleCloudPlatform/contact-center-ai-samples]                                    64.7308     12.267      5.277      0.000      40.688      88.774
C(pair_id)[T.BaihaiAI/IDP__vs__FirebaseExtended/firestore-nlp-extension]                                         -5.0000      1.182     -4.230      0.000      -7.317      -2.683
C(pair_id)[T.Ben